# 02 · Conditional Diffusion Training

Train a class-conditional UNet to predict the noise added at a random diffusion step. Sampling runs the reverse DDPM update from `t=T-1` to `t=0`. The same `DiffusionSchedule` object is used for both training (forward `q_sample`) and inference (`sample_images`).

In [ ]:
import sys
from pathlib import Path

ROOT = Path.cwd()
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

import torch
import matplotlib.pyplot as plt

from data.dataloader import get_mnist_loaders
from model.diffusion import ConditionalUNet
from training.train_diffusion import (
    linear_beta_schedule, cosine_beta_schedule,
    DiffusionSchedule, train_diffusion, sample_images,
)
from utils.visualize import plot_image_grid, plot_training_history

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('device:', device)

## Hyperparameters

In [ ]:
NUM_CLASSES = 10
BATCH_SIZE = 128
EPOCHS = 15            # 20-30 gives noticeably crisper digits
LR = 2e-4
NUM_TIMESTEPS = 1000
SCHEDULE = 'linear'    # try 'cosine' for the stand-out variant

## Build the data + model

In [ ]:
train_loader, _ = get_mnist_loaders(batch_size=BATCH_SIZE, num_workers=2)

unet = ConditionalUNet(num_classes=NUM_CLASSES, base_channels=64, time_dim=128).to(device)
print(f'UNet params: {sum(p.numel() for p in unet.parameters()):,}')

# Shape sanity check before training
x = torch.randn(4, 1, 28, 28, device=device)
t = torch.randint(0, NUM_TIMESTEPS, (4,), device=device)
y = torch.randint(0, NUM_CLASSES, (4,), device=device)
out = unet(x, t, y)
print('pred shape:', out.shape, '(should equal input shape)')

## Train

Training samples a random timestep `t` per example, adds the matching amount of Gaussian noise via `q_sample`, and minimizes MSE between the predicted noise and the actual noise.

In [ ]:
history = train_diffusion(
    unet,
    train_loader,
    epochs=EPOCHS,
    lr=LR,
    num_timesteps=NUM_TIMESTEPS,
    schedule=SCHEDULE,
    device=device,
    checkpoint_dir='checkpoints/diffusion',
    log_every=200,
)

In [ ]:
fig = plot_training_history(history, title='Diffusion MSE')
plt.show()

## Sample class-conditional digits

Run the reverse diffusion loop starting from pure Gaussian noise. We request 8 samples per class so each row in the grid corresponds to one digit identity.

In [ ]:
betas = linear_beta_schedule(NUM_TIMESTEPS) if SCHEDULE == 'linear' else cosine_beta_schedule(NUM_TIMESTEPS)
schedule = DiffusionSchedule.from_betas(betas).to(device)

n_per_class = 8
labels = torch.arange(NUM_CLASSES, device=device).repeat_interleave(n_per_class)
samples = sample_images(unet, labels, schedule).cpu()

fig = plot_image_grid(samples, labels=labels.cpu().tolist(), n_cols=n_per_class,
                     title='Diffusion: 8 samples per digit class')
plt.show()

## Bonus: visualize the denoising trajectory

Plot snapshots from the reverse process so you can see noise gradually resolving into a digit. We subsample 10 evenly spaced snapshots from the full T-step chain.

In [ ]:
label = torch.tensor([3], device=device)
traj = sample_images(unet, label, schedule, return_trajectory=True)
indices = torch.linspace(0, len(traj) - 1, 10).long().tolist()
snapshots = torch.stack([traj[i][0] for i in indices])
fig = plot_image_grid(snapshots, n_cols=10, title='Reverse diffusion (class 3, T=0 ... T=1000)')
plt.show()